In [ ]:
import os, json, time
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

load_dotenv(override=True)
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "paste-model-name-from-groq-console"

df = pd.read_csv("../data/test_sample_200.csv")
labels = sorted(df["category"].unique())          # the 10 intents

def classify(text):
    prompt = (
        "Classify the banking customer message into exactly one category.\n"
        f"Categories: {', '.join(labels)}\n"
        'Answer only with JSON: {"label": "<category>"}\n\n'
        f"Message: {text}"
    )
    start = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},   # force valid JSON
        temperature=0,                              # reproducible answers
    )
    seconds = time.time() - start
    try:
        label = json.loads(resp.choices[0].message.content)["label"]
    except (json.JSONDecodeError, KeyError, TypeError):
        label = "INVALID"
    if label not in labels:
        label = "INVALID"
    time.sleep(3)                                   # stay under free-tier limits
    return label, seconds, resp.usage.prompt_tokens, resp.usage.completion_tokens

In [ ]:
# quick check on 5 texts first
print([classify(t)[0] for t in df["text"].head(5)])